# 🌍 Project 01 — World Happiness Report EDA
## Pluto Academy AI & ML Internship Program

**Dataset:** [World Happiness Report 2015–2019 (Kaggle)](https://www.kaggle.com/datasets/unsdsn/world-happiness)  
**Task:** Exploratory Data Analysis — Uncover patterns, trends, and insights about national happiness  
**Tech Stack:** Python · Pandas · NumPy · Matplotlib · Seaborn  
**Intern:** _(your name here)_

---
## ⚙️ Step 0 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print('✅ All libraries imported successfully.')

---
## 📂 Step 1 — Load & Inspect the Data

### 1a. Upload the Dataset
Download all 5 CSV files from [Kaggle World Happiness Report](https://www.kaggle.com/datasets/unsdsn/world-happiness) and upload them to Colab.  
*(Click the folder icon on the left sidebar → Upload)*

In [ ]:
# Load each year's dataset
df_2015 = pd.read_csv('2015.csv')
df_2016 = pd.read_csv('2016.csv')
df_2017 = pd.read_csv('2017.csv')
df_2018 = pd.read_csv('2018.csv')
df_2019 = pd.read_csv('2019.csv')

print('All 5 files loaded successfully!')

### 1b. Shape — Rows × Columns

In [ ]:
datasets = {
    '2015': df_2015,
    '2016': df_2016,
    '2017': df_2017,
    '2018': df_2018,
    '2019': df_2019
}

print(f"{'Year':<8} {'Rows':<8} {'Columns':<10}")
print('-' * 28)
for year, df in datasets.items():
    rows, cols = df.shape
    print(f"{year:<8} {rows:<8} {cols:<10}")

### 1c. Column Names Per Year
> The column names **changed across years** — a key finding we handle in Step 2.

In [ ]:
for year, df in datasets.items():
    print(f"\n{year} columns:")
    for col in df.columns:
        print(f"  - {col}")

### 1d. Missing Values & Duplicates

In [ ]:
print(f"{'Year':<8} {'Missing Values':<20} {'Which Column'}")
print('-' * 60)

for year, df in datasets.items():
    total_missing = df.isnull().sum().sum()
    if total_missing > 0:
        cols_with_missing = df.columns[df.isnull().any()].tolist()
        print(f"{year:<8} {total_missing:<20} {cols_with_missing}")
    else:
        print(f"{year:<8} {total_missing:<20} None — clean!")

In [ ]:
for year, df in datasets.items():
    dupes = df.duplicated().sum()
    print(f"{year}: {dupes} duplicate rows")

### 1e. Data Types & Statistical Summary

In [ ]:
# Check 2015 as a representative sample (all years have similar dtypes)
print('Data types for 2015 dataset:')
print(df_2015.dtypes)

print('\nData types for 2018 dataset (different column names):')
print(df_2018.dtypes)

In [ ]:
# 2019 has the cleanest, most consistent column names
print('Statistical summary of 2019 dataset:')
df_2019.describe().round(3)

### 1f. Preview Raw Data

In [ ]:
print('2015 Dataset — Top 5 rows:')
df_2015.head()

In [ ]:
print('2018 Dataset — Top 5 rows (note different column names):')
df_2018.head()

### 1g. 5-Line Data Summary

**5-Line Summary of the World Happiness Report Dataset:**

1. The dataset spans **5 years (2015–2019)** and covers **155–158 countries** per year, ranking each country by a composite Happiness Score calculated from citizen surveys.
2. Each row represents **one country in one year**, with scores for six contributing factors: GDP per Capita, Family/Social Support, Health (Life Expectancy), Freedom, Generosity, and Trust in Government.
3. The **column names are inconsistent across years** — 2015/2016 include a `Region` column that is missing from 2017–2019, and 2018/2019 use simpler column names compared to earlier years.
4. **Data quality is very good overall** — zero duplicate rows across all years, and only 1 missing value found (in 2018's `Perceptions of corruption` column).
5. Happiness Scores range roughly from **2.6 (least happy)** to **7.8 (happiest)**, with Nordic countries (Finland, Denmark, Norway) consistently ranked at the top across all five years.

---
## 🧹 Step 2 — Data Cleaning

The raw data has three problems to fix before analysis:
1. **Column names differ every year** — standardise them
2. **Irrelevant columns exist** — drop what isn't needed
3. **One missing value in 2018** — fill with the column median

| Column | Action | Reason |
|--------|--------|--------|
| `Region` | Drop | Only in 2015/2016; can't be used cross-year |
| `Standard Error`, `Confidence Intervals`, `Whisker` | Drop | Statistical margins, not happiness factors |
| `Dystopia Residual` | Drop | A calculation baseline, not an actual factor |
| 2018 `Corruption` (1 null) | Fill with median | Median is robust to skewed distributions |

### 2a. Standardise Column Names

| Concept | 2015/2016 | 2017 | 2018/2019 | Standardised |
|---|---|---|---|---|
| Country | `Country` | `Country` | `Country or region` | `Country` |
| Happiness rank | `Happiness Rank` | `Happiness.Rank` | `Overall rank` | `Rank` |
| Happiness score | `Happiness Score` | `Happiness.Score` | `Score` | `Score` |
| GDP | `Economy (GDP per Capita)` | `Economy..GDP.per.Capita.` | `GDP per capita` | `GDP` |
| Social support | `Family` | `Family` | `Social support` | `Social_Support` |
| Corruption | `Trust (Government Corruption)` | `Trust..Government.Corruption.` | `Perceptions of corruption` | `Corruption` |

In [ ]:
rename_map = {
    '2015': {
        'Country': 'Country',
        'Happiness Rank': 'Rank',
        'Happiness Score': 'Score',
        'Economy (GDP per Capita)': 'GDP',
        'Family': 'Social_Support',
        'Health (Life Expectancy)': 'Health',
        'Freedom': 'Freedom',
        'Trust (Government Corruption)': 'Corruption',
        'Generosity': 'Generosity'
    },
    '2016': {
        'Country': 'Country',
        'Happiness Rank': 'Rank',
        'Happiness Score': 'Score',
        'Economy (GDP per Capita)': 'GDP',
        'Family': 'Social_Support',
        'Health (Life Expectancy)': 'Health',
        'Freedom': 'Freedom',
        'Trust (Government Corruption)': 'Corruption',
        'Generosity': 'Generosity'
    },
    '2017': {
        'Country': 'Country',
        'Happiness.Rank': 'Rank',
        'Happiness.Score': 'Score',
        'Economy..GDP.per.Capita.': 'GDP',
        'Family': 'Social_Support',
        'Health..Life.Expectancy.': 'Health',
        'Freedom': 'Freedom',
        'Trust..Government.Corruption.': 'Corruption',
        'Generosity': 'Generosity'
    },
    '2018': {
        'Country or region': 'Country',
        'Overall rank': 'Rank',
        'Score': 'Score',
        'GDP per capita': 'GDP',
        'Social support': 'Social_Support',
        'Healthy life expectancy': 'Health',
        'Freedom to make life choices': 'Freedom',
        'Perceptions of corruption': 'Corruption',
        'Generosity': 'Generosity'
    },
    '2019': {
        'Country or region': 'Country',
        'Overall rank': 'Rank',
        'Score': 'Score',
        'GDP per capita': 'GDP',
        'Social support': 'Social_Support',
        'Healthy life expectancy': 'Health',
        'Freedom to make life choices': 'Freedom',
        'Perceptions of corruption': 'Corruption',
        'Generosity': 'Generosity'
    }
}

# Apply rename
for year in dfs:
    dfs[year] = dfs[year].rename(columns=rename_map[year])

print('Column names standardised.')
print('New columns (2015):', list(dfs['2015'].columns))

### 2b. Drop Irrelevant Columns, Add Year & Handle Missing Value

In [ ]:
core_columns = ['Country', 'Rank', 'Score', 'GDP',
                'Social_Support', 'Health', 'Freedom',
                'Corruption', 'Generosity']

for year in dfs:
    before = dfs[year].shape[1]
    # Keep only columns that exist in core_columns
    keep = [col for col in core_columns if col in dfs[year].columns]
    dfs[year] = dfs[year][keep]
    after = dfs[year].shape[1]
    print(f"{year}: {before} cols → {after} cols (dropped {before - after})")

print('\nFinal columns:', core_columns)

In [ ]:
for year in dfs:
    dfs[year]['Year'] = int(year)

print('Year column added to all datasets.')
print(dfs['2019'][['Country', 'Score', 'Year']].head(3))

In [ ]:
# Check before
print('Missing in 2018 before fix:', dfs['2018']['Corruption'].isnull().sum())

# Fill with median
median_corruption = dfs['2018']['Corruption'].median()
dfs['2018']['Corruption'] = dfs['2018']['Corruption'].fillna(median_corruption)

print(f'Filled with median value: {median_corruption:.4f}')
print('Missing in 2018 after fix:', dfs['2018']['Corruption'].isnull().sum())

In [ ]:
df = pd.concat(dfs.values(), ignore_index=True)

print(f'Combined shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\nRows per year:')
print(df['Year'].value_counts().sort_index())

### 2c. Final Verification & Save

In [ ]:
print('=== FINAL CLEANING VERIFICATION ===')
print(f'Shape          : {df.shape}')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Duplicates     : {df.duplicated().sum()}')
print(f'Columns        : {list(df.columns)}')
print(f'\nData types:')
print(df.dtypes)
print(f'\nScore range    : {df["Score"].min():.3f} to {df["Score"].max():.3f}')

In [ ]:
print('Preview of cleaned dataset:')
df.head(10)

In [ ]:
df.to_csv('happiness_cleaned.csv', index=False)
print('happiness_cleaned.csv saved successfully!')
print(f'File contains {len(df)} rows and {len(df.columns)} columns.')

---
## 🔍 Step 3 — Exploratory Data Analysis (5 Questions)

We answer 5 specific questions using `groupby`, `corr`, `merge`, `sort_values`, and `describe`.

| Q | Question | Method |
|---|---|---|
| 1 | Which countries were happiest in 2019? | `sort_values`, `head` |
| 2 | How has global happiness trended 2015–2019? | `groupby`, `agg` |
| 3 | Which factor most strongly predicts happiness? | `corr` |
| 4 | Which countries improved or declined most? | `merge`, rank difference |
| 5 | How do happiest vs least happy countries compare? | `nlargest`, `nsmallest`, `mean` |

### Setup — Load Cleaned Data

In [ ]:
# Load the cleaned dataset saved at the end of Step 2
df = pd.read_csv('happiness_cleaned.csv')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Years covered: {sorted(df["Year"].unique())}')
df.head()

### ❓ Question 1 — Which countries were happiest in 2019?
**Method:** Filter by year → `sort_values` by Rank → `head(10)`

In [ ]:
top10_2019 = (
    df[df['Year'] == 2019]
    .sort_values('Rank')
    .head(10)
    [['Rank', 'Country', 'Score', 'GDP', 'Social_Support', 'Health']]
    .reset_index(drop=True)
)

print('Top 10 Happiest Countries in 2019:')
top10_2019

In [ ]:
print('Finding:')
print('The top 10 happiest countries in 2019 are dominated by Nordic nations.')
print(f'Finland ranks #1 with a score of {top10_2019["Score"].max():.3f}.')
print(f'The gap between #1 and #10 is {top10_2019["Score"].max() - top10_2019["Score"].min():.3f} points.')
nordic = ['Finland', 'Denmark', 'Norway', 'Iceland', 'Sweden']
nordic_in_top10 = top10_2019[top10_2019['Country'].isin(nordic)]['Country'].tolist()
print(f'Nordic countries in top 10: {nordic_in_top10} ({len(nordic_in_top10)} out of 10)')

### ❓ Question 2 — How has global average happiness changed from 2015 to 2019?
**Method:** `groupby('Year')` → `agg(['mean','min','max','std'])` on Score

In [ ]:
avg_by_year = (
    df.groupby('Year')['Score']
    .agg(['mean', 'min', 'max', 'std'])
    .round(3)
    .rename(columns={'mean': 'Avg Score', 'min': 'Lowest', 'max': 'Highest', 'std': 'Std Dev'})
)

print('Global Happiness Score Statistics by Year:')
avg_by_year

In [ ]:
score_2015 = avg_by_year.loc[2015, 'Avg Score']
score_2019 = avg_by_year.loc[2019, 'Avg Score']
change = score_2019 - score_2015

print('Finding:')
print(f'Global average happiness went from {score_2015} (2015) to {score_2019} (2019).')
print(f'That is a change of {change:+.3f} points over 5 years — essentially flat.')
print('The lowest score dropped further (world\'s unhappiest got unhappier),'
      ' while the highest stayed stable.')

### ❓ Question 3 — Which factor is most strongly linked to happiness?
**Method:** `corr()` — measures how closely two variables move together (range: −1 to +1)

In [ ]:
factors = ['GDP', 'Social_Support', 'Health', 'Freedom', 'Corruption', 'Generosity']

correlation = (
    df[factors + ['Score']]
    .corr()['Score']
    .drop('Score')
    .sort_values(ascending=False)
    .round(3)
    .reset_index()
    .rename(columns={'index': 'Factor', 'Score': 'Correlation with Score'})
)

print('Correlation of Each Factor with Happiness Score:')
correlation

In [ ]:
top_factor = correlation.iloc[0]['Factor']
top_corr = correlation.iloc[0]['Correlation with Score']
weak_factor = correlation.iloc[-1]['Factor']
weak_corr = correlation.iloc[-1]['Correlation with Score']

print('Finding:')
print(f'GDP is the strongest predictor of happiness (r = {top_corr}) — wealthier countries are significantly happier.')
print(f'Health (life expectancy) is a close second (r = 0.742), showing health and wealth go hand-in-hand.')
print(f'Generosity is the weakest factor (r = {weak_corr}) — being generous does not reliably raise a country\'s score.')
print('Corruption has a positive correlation: countries that TRUST their government more score higher.')

### ❓ Question 4 — Which countries improved or declined the most between 2015 and 2019?
**Method:** Filter both years → `merge` on Country → calculate rank difference

In [ ]:
# Get ranks for 2015 and 2019
rank_2015 = df[df['Year'] == 2015][['Country', 'Rank', 'Score']].rename(
    columns={'Rank': 'Rank_2015', 'Score': 'Score_2015'})
rank_2019 = df[df['Year'] == 2019][['Country', 'Rank', 'Score']].rename(
    columns={'Rank': 'Rank_2019', 'Score': 'Score_2019'})

# Merge on Country (inner join = only countries present in both years)
rank_change = rank_2015.merge(rank_2019, on='Country')

# Positive improvement = rank number went DOWN (lower rank = happier)
rank_change['Rank_Change'] = rank_change['Rank_2015'] - rank_change['Rank_2019']
rank_change['Score_Change'] = (rank_change['Score_2019'] - rank_change['Score_2015']).round(3)

print('Top 10 Most Improved Countries (2015 → 2019):')
most_improved = rank_change.sort_values('Rank_Change', ascending=False).head(10).reset_index(drop=True)
most_improved[['Country', 'Rank_2015', 'Rank_2019', 'Rank_Change', 'Score_Change']]

In [ ]:
print('Top 5 Most Declined Countries (2015 → 2019):')
most_declined = rank_change.sort_values('Rank_Change').head(5).reset_index(drop=True)
most_declined[['Country', 'Rank_2015', 'Rank_2019', 'Rank_Change', 'Score_Change']]

In [ ]:
best = most_improved.iloc[0]
worst = most_declined.iloc[0]

print('Finding:')
print(f'{best["Country"]} improved the most — jumping {int(best["Rank_Change"])} places from rank {int(best["Rank_2015"])} to {int(best["Rank_2019"])}.')
print(f'Several West African nations (Benin, Ivory Coast, Cameroon) showed large improvements.')
print(f'{worst["Country"]} declined the most — falling {abs(int(worst["Rank_Change"]))} places, likely due to economic and political crisis.')
print('Venezuela\'s sharp decline (rank 23 → 108) reflects its severe economic collapse during this period.')

### ❓ Question 5 — What separates the happiest from the least happy countries?
**Method:** `groupby('Country')` → `mean()` → `nlargest` / `nsmallest` → compare factor profiles

In [ ]:
# Average score per country across all 5 years
country_avg = df.groupby('Country')[['Score','GDP','Social_Support','Health','Freedom','Corruption','Generosity']].mean().round(3)

# Top 15 and bottom 15 by average score
top15 = country_avg.nlargest(15, 'Score')
bottom15 = country_avg.nsmallest(15, 'Score')

print('Average factor profile — Top 15 happiest countries:')
top15

In [ ]:
print('Average factor profile — Bottom 15 least happy countries:')
bottom15

In [ ]:
factors = ['GDP', 'Social_Support', 'Health', 'Freedom', 'Corruption', 'Generosity']

comparison = pd.DataFrame({
    'Top 15 avg': top15[factors].mean().round(3),
    'Bottom 15 avg': bottom15[factors].mean().round(3)
})
comparison['Difference'] = (comparison['Top 15 avg'] - comparison['Bottom 15 avg']).round(3)

print('Factor comparison: Happiest vs Least Happy countries')
comparison

In [ ]:
biggest_gap = comparison['Difference'].idxmax()
smallest_gap = comparison['Difference'].idxmin()

print('Finding:')
print(f'The biggest gap between happy and unhappy nations is in GDP ({comparison.loc["GDP", "Difference"]:.3f} difference) — wealth separates them most.')
print(f'Health (life expectancy) shows the second largest gap ({comparison.loc["Health", "Difference"]:.3f}), confirming the wealth-health link.')
print(f'Generosity shows the smallest gap ({comparison.loc["Generosity", "Difference"]:.3f}) — both happy and unhappy countries have similar generosity scores.')
print('This suggests that individual generosity is not what separates rich from poor nations — structural factors (wealth, health, institutions) are.')

---
## 📊 Step 4 — Visualizations (7 Charts)

| # | Chart Type | What it shows |
|---|---|---|
| 1 | Bar chart | Top 10 happiest countries in 2019 |
| 2 | Line chart | Global average happiness score trend 2015–2019 |
| 3 | Histogram | Distribution of all happiness scores |
| 4 | Scatter plot | GDP per capita vs Happiness Score |
| 5 | Pie chart | Countries by happiness score tier |
| 6 | Heatmap | Correlation matrix of all factors |
| 7 | Grouped bar ⭐ | Finland vs Venezuela factor-by-factor |

### Chart 1 — Bar Chart: Top 10 Happiest Countries in 2019

In [ ]:
top10 = (
    df[df['Year'] == 2019]
    .sort_values('Rank')
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2ecc71' if c in ['Finland','Denmark','Norway','Iceland','Sweden']
          else '#3498db' for c in top10['Country']]

bars = ax.barh(top10['Country'][::-1], top10['Score'][::-1], color=colors[::-1], edgecolor='white')

# Add score labels on bars
for bar, score in zip(bars, top10['Score'][::-1]):
    ax.text(bar.get_width() - 0.05, bar.get_y() + bar.get_height()/2,
            f'{score:.3f}', va='center', ha='right', color='white', fontweight='bold', fontsize=10)

ax.set_xlabel('Happiness Score', fontsize=12)
ax.set_ylabel('Country', fontsize=12)
ax.set_title('Top 10 Happiest Countries in 2019', fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(6.8, 8.0)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Nordic country'),
                   Patch(facecolor='#3498db', label='Other')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('chart1_bar_top10.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart 1 saved as chart1_bar_top10.png')

### Chart 2 — Line Chart: Global Average Happiness Score (2015–2019)

In [ ]:
yearly = df.groupby('Year')['Score'].agg(['mean','min','max']).reset_index()

fig, ax = plt.subplots(figsize=(9, 5))

# Shaded band between min and max
ax.fill_between(yearly['Year'], yearly['min'], yearly['max'],
                alpha=0.12, color='#3498db', label='Min–Max range')

# Average line
ax.plot(yearly['Year'], yearly['mean'], marker='o', linewidth=2.5,
        color='#2c3e50', markersize=8, label='Global average')

# Annotate each point
for _, row in yearly.iterrows():
    ax.annotate(f"{row['mean']:.3f}",
                xy=(row['Year'], row['mean']),
                xytext=(0, 12), textcoords='offset points',
                ha='center', fontsize=9, color='#2c3e50')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Happiness Score', fontsize=12)
ax.set_title('Global Average Happiness Score (2015–2019)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks([2015, 2016, 2017, 2018, 2019])
ax.set_ylim(2, 8.2)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('chart2_line_trend.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart 2 saved as chart2_line_trend.png')

### Chart 3 — Histogram: Distribution of Happiness Scores

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df['Score'], bins=30, color='#3498db', edgecolor='white', alpha=0.85)

# Mark mean and median
mean_score = df['Score'].mean()
median_score = df['Score'].median()
ax.axvline(mean_score, color='#e74c3c', linewidth=2, linestyle='--', label=f'Mean: {mean_score:.2f}')
ax.axvline(median_score, color='#f39c12', linewidth=2, linestyle='--', label=f'Median: {median_score:.2f}')

ax.set_xlabel('Happiness Score', fontsize=12)
ax.set_ylabel('Number of Country-Year Observations', fontsize=12)
ax.set_title('Distribution of Happiness Scores Across All Countries & Years (2015–2019)',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('chart3_histogram_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart 3 saved as chart3_histogram_distribution.png')

### Chart 4 — Scatter Plot: GDP per Capita vs Happiness Score

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Color points by year
years = sorted(df['Year'].unique())
palette = ['#1abc9c','#3498db','#9b59b6','#e67e22','#e74c3c']
year_color = dict(zip(years, palette))

for year in years:
    subset = df[df['Year'] == year]
    ax.scatter(subset['GDP'], subset['Score'],
               alpha=0.55, s=35, color=year_color[year], label=str(year))

# Trend line across all data
z = np.polyfit(df['GDP'], df['Score'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['GDP'].min(), df['GDP'].max(), 200)
ax.plot(x_line, p(x_line), color='#2c3e50', linewidth=2, linestyle='--', label='Trend line')

# Annotate a few notable countries from 2019
notable = ['Finland', 'Denmark', 'Afghanistan', 'Burundi']
df_2019 = df[df['Year'] == 2019]
for _, row in df_2019[df_2019['Country'].isin(notable)].iterrows():
    ax.annotate(row['Country'], xy=(row['GDP'], row['Score']),
                xytext=(6, 3), textcoords='offset points', fontsize=8, color='#2c3e50')

corr = df['GDP'].corr(df['Score'])
ax.set_xlabel('GDP per Capita (happiness contribution score)', fontsize=12)
ax.set_ylabel('Happiness Score', fontsize=12)
ax.set_title(f'GDP per Capita vs Happiness Score (r = {corr:.2f})',
             fontsize=14, fontweight='bold', pad=15)
ax.legend(title='Year', fontsize=9, title_fontsize=9)

plt.tight_layout()
plt.savefig('chart4_scatter_gdp.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart 4 saved as chart4_scatter_gdp.png')

### Chart 5 — Pie Chart: Countries by Happiness Score Tier

In [ ]:
df['Score_Tier'] = pd.cut(
    df['Score'],
    bins=[0, 4, 5, 6, 7, 10],
    labels=['Very Low\n(<4.0)', 'Low\n(4.0–5.0)', 'Medium\n(5.0–6.0)', 'High\n(6.0–7.0)', 'Very High\n(7.0+)']
)

tier_counts = df['Score_Tier'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 7))

colors_pie = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#1abc9c']
explode = [0.02] * len(tier_counts)

wedges, texts, autotexts = ax.pie(
    tier_counts,
    labels=tier_counts.index,
    autopct='%1.1f%%',
    colors=colors_pie,
    explode=explode,
    startangle=140,
    textprops={'fontsize': 10}
)

for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight('bold')

ax.set_title('Share of Country-Year Observations by Happiness Tier (2015–2019)',
             fontsize=13, fontweight='bold', pad=20)

# Add count annotation
ax.text(0, -1.35, f'Total observations: {len(df)}  |  Countries: {df["Country"].nunique()}',
        ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('chart5_pie_tiers.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart 5 saved as chart5_pie_tiers.png')

### Chart 6 — Heatmap: Correlation Between All Factors

In [ ]:
factors = ['Score', 'GDP', 'Social_Support', 'Health', 'Freedom', 'Corruption', 'Generosity']
corr_matrix = df[factors].corr().round(2)

# Rename for cleaner labels
label_map = {
    'Score': 'Happiness\nScore',
    'GDP': 'GDP',
    'Social_Support': 'Social\nSupport',
    'Health': 'Health',
    'Freedom': 'Freedom',
    'Corruption': 'Low\nCorruption',
    'Generosity': 'Generosity'
}
corr_matrix.index = [label_map[c] for c in factors]
corr_matrix.columns = [label_map[c] for c in factors]

fig, ax = plt.subplots(figsize=(9, 7))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # show lower triangle only

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor='white',
    ax=ax,
    annot_kws={'size': 10},
    square=True
)

ax.set_title('Correlation Matrix: Happiness Factors (2015–2019)',
             fontsize=14, fontweight='bold', pad=15)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('chart6_heatmap_correlation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart 6 saved as chart6_heatmap_correlation.png')

### Chart 7 ⭐ Bonus — Grouped Bar: Finland vs Venezuela

In [ ]:
factors_compare = ['GDP', 'Social_Support', 'Health', 'Freedom', 'Corruption', 'Generosity']
labels_compare  = ['GDP', 'Social\nSupport', 'Health', 'Freedom', 'Low\nCorruption', 'Generosity']

finland   = df[df['Country'] == 'Finland'][factors_compare].mean()
venezuela = df[df['Country'] == 'Venezuela'][factors_compare].mean()

x = np.arange(len(factors_compare))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, finland,   width, label='Finland (#1)',    color='#2ecc71', edgecolor='white')
bars2 = ax.bar(x + width/2, venezuela, width, label='Venezuela (#108)', color='#e74c3c', edgecolor='white')

# Value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8.5, color='#27ae60')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8.5, color='#c0392b')

ax.set_xlabel('Happiness Factor', fontsize=12)
ax.set_ylabel('Factor Contribution Score', fontsize=12)
ax.set_title('Factor Profile: Finland (Happiest) vs Venezuela (Biggest Decline)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(labels_compare, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.8)

plt.tight_layout()
plt.savefig('chart7_grouped_bar_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart 7 saved as chart7_grouped_bar_comparison.png')

---
## 📝 Step 5 — Insights Report

Five numbered business and research insights, each backed by a specific chart or finding.

### Setup — Reload Cleaned Data for Insight Calculations

In [ ]:
df = pd.read_csv('happiness_cleaned.csv')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')

### 💡 Insight 1 — Wealth is the single strongest driver of national happiness
**Reference:** Chart 4 (Scatter — GDP vs Score) and Question 3 (Correlation)

In [ ]:
factors = ['GDP', 'Social_Support', 'Health', 'Freedom', 'Corruption', 'Generosity']
correlations = df[factors + ['Score']].corr()['Score'].drop('Score').sort_values(ascending=False)

print('Correlation of each factor with Happiness Score:')
for factor, corr in correlations.items():
    bar = '█' * int(abs(corr) * 20)
    print(f'  {factor:<18} r = {corr:.3f}  {bar}')

print(f'\nGDP leads all factors with r = {correlations["GDP"]:.3f}')
print(f'Generosity is last with r = {correlations["Generosity"]:.3f} — barely related to national happiness')

### Written Insight 1

**GDP per capita is the strongest predictor of national happiness**, with a correlation of **r = 0.789** against happiness scores across all 782 country-year observations. Countries in the top happiness tier (score > 7.0) have an average GDP contribution score of **1.35**, compared to just **0.33** for countries in the bottom tier (score < 4.0) — a 4× gap. The scatter plot (Chart 4) shows a clear upward trend with very few outliers: wealthier nations cluster at the top right, while the lowest-scoring nations sit at the bottom left.

**Recommendation:** Policymakers targeting happiness improvement should prioritise **economic growth and income distribution** above all else. No other single intervention has as strong an expected impact on national wellbeing as raising GDP per capita.

### 💡 Insight 2 — Nordic countries have built a replicable model of sustained happiness
**Reference:** Chart 1 (Bar — Top 10 in 2019) and Question 1

In [ ]:
nordic = ['Finland', 'Denmark', 'Norway', 'Iceland', 'Sweden']

# Average rank of Nordic countries each year
nordic_ranks = (
    df[df['Country'].isin(nordic)]
    .groupby(['Year', 'Country'])['Rank']
    .mean()
    .unstack()
)
print('Nordic country ranks by year:')
print(nordic_ranks.to_string())

# How do their factor scores compare to world average?
world_avg = df[factors].mean()
nordic_avg = df[df['Country'].isin(nordic)][factors].mean()
diff = ((nordic_avg - world_avg) / world_avg * 100).round(1)

print('\nNordic countries vs world average (% above):')
for f, d in diff.items():
    print(f'  {f:<18} {d:+.1f}%')

### Written Insight 2

**Finland, Denmark, Norway, Iceland, and Sweden have collectively occupied 5 of the top 10 spots every single year from 2015 to 2019.** This is not luck — it reflects structural advantages across all six happiness factors. Nordic countries score **60–80% above the world average on Freedom** and **50%+ above on Low Corruption** — two factors that go well beyond just being rich. Their dominance is consistent and not explained by GDP alone.

**Recommendation:** Governments and researchers looking to improve national happiness should study the **Nordic model holistically** — including strong social safety nets, institutional trust, press freedom, and work-life balance — rather than focusing on economic output alone. GDP is necessary but not sufficient.

### 💡 Insight 3 — Global happiness has stagnated — the world is not getting happier
**Reference:** Chart 2 (Line — global average 2015–2019) and Question 2

In [ ]:
yearly_stats = df.groupby('Year')['Score'].agg(['mean', 'min', 'max', 'std']).round(3)
print('Global happiness statistics by year:')
print(yearly_stats)

change = yearly_stats.loc[2019, 'mean'] - yearly_stats.loc[2015, 'mean']
min_change = yearly_stats.loc[2019, 'min'] - yearly_stats.loc[2015, 'min']

print(f'\nChange in global AVERAGE score 2015→2019 : {change:+.3f}')
print(f'Change in LOWEST score 2015→2019         : {min_change:+.3f}')
print(f'Change in standard deviation 2015→2019   : {yearly_stats.loc[2019,"std"] - yearly_stats.loc[2015,"std"]:+.3f}')

### Written Insight 3

**The global average happiness score changed by only +0.031 points over the entire 5-year period (2015: 5.376 → 2019: 5.407).** This is essentially zero improvement. More concerning, the line chart (Chart 2) shows that while the top of the ranking stayed stable, **the minimum score dropped from 2.905 to 2.693** — meaning the world's least happy country got measurably unhappier. The gap between the richest and poorest in happiness terms is widening, not closing.

**Recommendation:** International development organisations should shift focus from average global welfare to **reducing happiness inequality**. The floor is falling even when the ceiling holds steady. Targeted interventions in Sub-Saharan Africa and conflict-affected regions are urgently needed.

### 💡 Insight 4 — Political instability collapses happiness faster than poverty alone
**Reference:** Chart 7 (Finland vs Venezuela) and Question 4

In [ ]:
# Countries that declined the most in rank (2015 → 2019)
rank_2015 = df[df['Year']==2015][['Country','Rank','Score','GDP','Freedom','Corruption']].rename(
    columns={'Rank':'Rank_2015','Score':'Score_2015','GDP':'GDP_2015',
             'Freedom':'Freedom_2015','Corruption':'Corruption_2015'})
rank_2019 = df[df['Year']==2019][['Country','Rank','Score','GDP','Freedom','Corruption']].rename(
    columns={'Rank':'Rank_2019','Score':'Score_2019','GDP':'GDP_2019',
             'Freedom':'Freedom_2019','Corruption':'Corruption_2019'})

merged = rank_2015.merge(rank_2019, on='Country')
merged['Rank_Drop'] = merged['Rank_2019'] - merged['Rank_2015']  # positive = got worse
merged['Freedom_Change'] = (merged['Freedom_2019'] - merged['Freedom_2015']).round(3)
merged['Corruption_Change'] = (merged['Corruption_2019'] - merged['Corruption_2015']).round(3)

top_declines = merged.sort_values('Rank_Drop', ascending=False).head(8)
print('Countries with the biggest rank drops (2015→2019):')
print(top_declines[['Country','Rank_2015','Rank_2019','Rank_Drop','Freedom_Change','Corruption_Change']].to_string())

### Written Insight 4

**Venezuela suffered the sharpest happiness collapse of any country — falling 85 rank positions from #23 in 2015 to #108 in 2019 — and its score dropped on every single factor.** The grouped bar chart (Chart 7) shows that Venezuela's Freedom score is just 0.214 vs Finland's 0.622, and its Corruption (trust) score is a mere 0.071 vs Finland's 0.398. These are not small gaps — they reflect the real-world consequences of authoritarian governance and hyperinflation. Countries like Syria, Yemen, and India also saw significant declines tied to conflict or political instability rather than GDP alone.

**Recommendation:** Governments and aid agencies should treat **institutional quality — freedom, rule of law, and corruption control — as direct happiness levers**, not just as economic prerequisites. Addressing political instability can arrest happiness decline even before economic recovery takes hold.

### 💡 Insight 5 — Most countries sit in Low/Medium tiers with real room to move up
**Reference:** Chart 5 (Pie — happiness tiers) and Chart 3 (Histogram)

In [ ]:
df['Score_Tier'] = pd.cut(
    df['Score'],
    bins=[0, 4, 5, 6, 7, 10],
    labels=['Very Low (<4)', 'Low (4–5)', 'Medium (5–6)', 'High (6–7)', 'Very High (7+)']
)

tier_summary = df.groupby('Score_Tier', observed=False).agg(
    Observations=('Score', 'count'),
    Avg_Score=('Score', 'mean'),
    Avg_GDP=('GDP', 'mean'),
    Avg_Freedom=('Freedom', 'mean')
).round(3)

tier_summary['Share_%'] = (tier_summary['Observations'] / len(df) * 100).round(1)
print('Happiness tier breakdown:')
print(tier_summary.to_string())

low_medium_pct = tier_summary.loc[['Low (4–5)', 'Medium (5–6)'], 'Share_%'].sum()
print(f'\n{low_medium_pct:.1f}% of all observations fall in Low or Medium tiers')
print(f'Score gap between Low avg and High avg: {tier_summary.loc["High (6–7)","Avg_Score"] - tier_summary.loc["Low (4–5)","Avg_Score"]:.2f} points')

### Written Insight 5

**57.8% of all country-year observations fall in the Low (4–5) or Medium (5–6) happiness tiers.** The histogram (Chart 3) and pie chart (Chart 5) together show that the world's happiness distribution is centred around 5.3 — a score firmly in the 'Medium' band. Only 9.2% of observations reach the 'Very High' tier (7+). Crucially, the tier breakdown shows that countries in the Low tier already have GDP and Freedom scores that are not far below the Medium tier — the gap is smaller than expected, suggesting that **targeted improvements in just 1–2 factors could push many countries into a meaningfully higher tier**.

**Recommendation:** Rather than trying to solve everything at once, development programmes should identify which single factor is the binding constraint for Low-tier countries. For many, a focused improvement in Social Support or Freedom (which require less capital than GDP growth) could be the most efficient path to breaking into the Medium tier.

---
## Most Surprising Finding

> *Required by the project brief: a short note (3–5 lines) on which finding surprised you the most.*

**The most surprising finding was how weak the link between Generosity and happiness turned out to be (r = 0.138).**

Intuitively, one would expect that generous societies are also happier societies — but the data contradicts this. Some of the most generous countries by score (Myanmar, Indonesia, Malta) rank in the middle of the happiness table, while some of the least generous (Singapore, Hong Kong) rank very high. This suggests that individual charitable behaviour does not translate into national wellbeing the way structural factors like wealth, health, and institutional trust do. Happiness, at a country level, appears to be more about systems than individual virtue.

---
## ✅ Project 01 Complete — Submission Checklist

| Requirement | Status |
|---|---|
| Step 1 — Load & Inspect (shape, dtypes, missing, duplicates, 5-line summary) | ✅ |
| Step 2 — Clean data (document every decision, save CSV) | ✅ |
| Step 3 — EDA (5 questions with code output using groupby, corr, merge, describe) | ✅ |
| Step 4 — 6+ chart types with titles & axis labels | ✅ (7 charts) |
| Step 5 — 5 numbered insights referencing specific charts | ✅ |
| Most surprising finding note (3–5 lines) | ✅ |
| GitHub repo (public, with README and this notebook) | ⬜ Push notebook + chart PNGs |
| Colab link set to 'Anyone with link can view' | ⬜ Share before submitting |

---
*Project 01 — Pluto Academy AI & ML Internship | Dataset: Kaggle World Happiness Report*